In [ ]:
# %pip install pydantic
# %pip install datamodel-code-generator

Let's say we have an expacted data schema of
```
{
    "name": string,
    "age": integer,
    "address": {
        "city": string,
        "zip_code": string,
        "number": integer
    }
}
```

In [1]:
from pydantic import BaseModel


class Address(BaseModel):
    """
    Cat API Address definition
    """
    city: str
    zip_code: str
    number: int

class CatRequest(BaseModel):
    """
    Cat API Request definition
    """
    name: str
    age: int
    address: Address

In [2]:
my_json = {
    "name": "Lévy",
    "age": 3,
    "address": {
        "city": "Wonderland",
        "zip_code": "ABCDE",
        "number": 123
    }
}

data = CatRequest.parse_obj(my_json)
data.name, data.address.number

('Lévy', 123)

In [3]:
from pydantic import ValidationError

bad_data = {
    "name": "Lévy",
    "age": "am I an age?",  # Note the type change here
    "address": {
        "city": "Wonderland",
        "zip_code": "ABCDE",
        "number": 123
    }
}

try:
    CatRequest.parse_obj(bad_data)
except ValidationError as err:
    print("Something went wrong with the data!")

Something went wrong with the data!


### Note that extra fields don't get caught

In [4]:
unnecessary_data = {
    "name": "Lévy",
    "age": 3,
    "key": "value",  # unnecessary
    "key2": "value2",  # unnecessary x2
    "address": {
        "city": "Wonderland",
        "zip_code": "ABCDE",
        "number": 123
    }
}

data = CatRequest.parse_obj(unnecessary_data)

In [5]:
from pydantic import BaseModel, Extra, Field


class Address(BaseModel):
    """
    Cat API Address definition
    """
    class Config:
        extra = Extra.forbid

    # Note how we can even add descriptions to the fields!
    city: str = Field(..., description="Where the cat lives")
    zip_code: str
    number: int

 
class CatRequest(BaseModel):
    """
    Cat API Request definition
    """
    class Config:
        extra = Extra.forbid

    name: str
    age: int
    address: Address

In [6]:
data = CatRequest.parse_obj(unnecessary_data)

ValidationError: 2 validation errors for CatRequest
key
  extra fields not permitted (type=value_error.extra)
key2
  extra fields not permitted (type=value_error.extra)

In `pydantic_cat.json`, we are defining our CatRequest schema by providing not only its properties: name, age, and address, but also with the ability to write down the nested definitions in one go.
Two valuable characteristics of JSON Schemas are:
The ability to select which fields are mandatory and optional by passing an array of `required` field names. For the sake of the explanation, we just chose name as a must-have.
Using `additionalProperties`, developers can easily control that these definitions do not become key-value dumps. Defining a contract is only helpful if we can ensure that it holds.

In [11]:
!datamodel-codegen --input pydantic_cat.json --input-file-type jsonschema --output pydantic_cat.py

In [13]:
with open('pydantic_cat.py', 'r') as f:
    print(f.read())

# generated by datamodel-codegen:
#   filename:  pydantic_cat.json
#   timestamp: 2022-04-17T22:15:23+00:00

from __future__ import annotations

from typing import Optional

from pydantic import BaseModel, Extra, Field


class Address(BaseModel):
    class Config:
        extra = Extra.forbid

    city: Optional[str] = Field(None, description="Cat's city")
    zip_code: Optional[str] = Field(None, description='Postal code')
    number: Optional[int] = Field(None, description='House number')


class CatRequest(BaseModel):
    class Config:
        extra = Extra.forbid

    name: str = Field(..., description="Cat's name.")
    age: Optional[int] = Field(None, description="Cat's age, in cat years.")
    address: Optional[Address] = Field(None, description='Where does the cat live.')



### Using `functools.singledispatch` for fucntion overriding
Note that `singledispatch` only considers the type of the first argument

In [14]:
from functools import singledispatch

@singledispatch
def process(model):
    """
    Default processing definition
    """
    raise NotImplementedError(f"I don't know how to process {type(model)}")

@process.register
def _(model: Address):
    """
    Handle addresses
    """
    print(f"Just got an address from {model.city}")

@process.register
def _(model: CatRequest):
    """
    Handle Cat Requests
    """
    print(f"Processing {model.name} the cat!")

address = Address(
    city="Wonderland",
    zip_code="ABCDE",
    number=123,
)
cat = CatRequest(
    name="Lévy",
    age=3,
    address=address
)

process(address)  # Just got an address from Wonderland
process(cat)  # Processing Lévy the cat!
process("something else")  # NotImplementedError: I don't know how to process <class 'str'>

Just got an address from Wonderland
Processing Lévy the cat!


NotImplementedError: I don't know how to process <class 'str'>